# Inspect control-prefix tokenization

Sample one shuffled FineWeb document for every bit count from 1 through 10. Tokens marked with `*` overlap the random bit value.

In [ ]:
import random
import sys
from pathlib import Path

from transformers import AutoTokenizer

start = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()
repo_root = next(path for path in [start, *start.parents] if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(repo_root))
from ciphers.kirchenbauer_et_al.binary_classification_mvp.data import compile_prefix, load_fineweb

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Base")
none_prefix = compile_prefix(None)
documents = [row["text"].removeprefix(none_prefix) for row in load_fineweb(None).take(10)]
rng = random.Random(42)

/mnt/align4_drive2/adrianoh/miniconda-installation/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name '__file__' is not defined

In [ ]:
def print_token_boundaries(prefix: str, bits: str) -> None:
    encoded = tokenizer(prefix, add_special_tokens=False, return_offsets_mapping=True)
    bit_start = prefix.index("<encoding_value> ") + len("<encoding_value> ")
    bit_end = bit_start + len(bits)
    pieces = []
    for index, (token_id, (start, end)) in enumerate(zip(encoded["input_ids"], encoded["offset_mapping"])):
        overlaps_bits = start < bit_end and end > bit_start
        if overlaps_bits:
            pieces.append(prefix[max(start, bit_start) : min(end, bit_end)])
        marker = "*" if overlaps_bits else " "
        token = tokenizer.convert_ids_to_tokens(token_id)
        print(f"{marker} {index:>2}  chars[{start:>2}:{end:<2}]  {prefix[start:end]!r:<22} id={token_id:<7} token={token!r}")
    print("bit token pieces:", pieces)

In [ ]:
for bit_count, document in enumerate(documents, start=1):
    bits = "".join(rng.choice("01") for _ in range(bit_count))
    prefix = compile_prefix(bits)
    print(f"\n{'=' * 80}\n{bit_count} bit(s): {bits}\nFineWeb preview: {document[:120]!r}\nPrefix: {prefix!r}")
    print_token_boundaries(prefix, bits)